# AI_FILTER — Surfacing Actionable Customer Feedback

## Use Case Overview

`AI_FILTER` evaluates each row against a natural-language condition and returns only the rows where the condition is true. Think of it as a `WHERE` clause that understands meaning, not just exact matches.

**SA use case:** From thousands of mixed customer feedback items, instantly surface only those that require action — escalations, bugs, billing issues, SLA violations — without building a complex classifier or reading every record manually.

**Dataset:** 20 synthetic customer feedback records stored as VARIANT JSON. Each contains free-text feedback, a rating, and user tier metadata.

**What we'll demonstrate:**
1. Basic AI_FILTER to find actionable feedback
2. Chaining AI_FILTER with AI_CLASSIFY to categorise what was filtered
3. Filtering specifically for enterprise-tier escalations
4. Building an "urgent inbox" view

In [ ]:
import os
import pandas as pd
import snowflake.connector

conn = snowflake.connector.connect(
    connection_name=os.getenv("SNOWFLAKE_CONNECTION_NAME") or "genai-labs"
)
conn.cursor().execute("USE DATABASE GENAI_LEARNING")
conn.cursor().execute("USE SCHEMA PUBLIC")
print("Connected:", conn.account)

## Step 2 — Data Exploration

In [ ]:
all_df = pd.read_sql("""
    SELECT
        feedback_id,
        product_name,
        channel,
        feedback:text::STRING    AS feedback_text,
        feedback:rating::INTEGER AS rating,
        feedback:user_tier::STRING AS user_tier
    FROM customer_feedback
    ORDER BY feedback_id
""", conn)
print(f"Total feedback: {len(all_df)}")
all_df

## Step 3 — AI_FILTER: Find Actionable Feedback

`AI_FILTER(text, condition)` returns TRUE/FALSE per row. Use it in a `WHERE` clause to keep only the rows that match a semantic condition.

The condition is a natural-language statement — not SQL syntax.

In [ ]:
actionable_df = pd.read_sql("""
    SELECT
        feedback_id,
        product_name,
        channel,
        feedback:text::STRING    AS feedback_text,
        feedback:rating::INTEGER AS rating,
        feedback:user_tier::STRING AS user_tier
    FROM customer_feedback
    WHERE AI_FILTER(
        feedback:text::STRING,
        'The feedback describes a technical bug, service outage, billing error, or urgent issue that requires immediate action from the product or support team'
    )
    ORDER BY rating ASC
""", conn)
print(f"Actionable feedback: {len(actionable_df)} of {len(all_df)} total ({len(actionable_df)/len(all_df)*100:.0f}%)")
actionable_df

## Step 4 — Chain AI_FILTER + AI_CLASSIFY

Filter to actionable rows, then classify each one into an issue type for routing.

In [ ]:
routed_df = pd.read_sql("""
    SELECT
        feedback_id,
        product_name,
        feedback:user_tier::STRING AS user_tier,
        feedback:text::STRING AS feedback_text,
        AI_CLASSIFY(
            feedback:text::STRING,
            ['bug_report', 'billing_issue', 'performance_degradation', 'missing_feature', 'sla_violation']
        ):label::STRING AS issue_type,
        AI_CLASSIFY(
            feedback:text::STRING,
            ['bug_report', 'billing_issue', 'performance_degradation', 'missing_feature', 'sla_violation']
        ):score::FLOAT AS confidence
    FROM customer_feedback
    WHERE AI_FILTER(
        feedback:text::STRING,
        'The feedback describes a technical bug, service outage, billing error, or urgent issue that requires immediate action from the product or support team'
    )
    ORDER BY confidence DESC
""", conn)
routed_df

## Step 5 — Enterprise Escalation Inbox

Build a priority inbox: actionable issues from enterprise customers only.

In [ ]:
escalations_df = pd.read_sql("""
    SELECT
        feedback_id,
        product_name,
        channel,
        feedback:user_tier::STRING AS user_tier,
        feedback:rating::INTEGER AS rating,
        feedback:text::STRING AS feedback_text
    FROM customer_feedback
    WHERE feedback:user_tier::STRING = 'enterprise'
      AND AI_FILTER(
            feedback:text::STRING,
            'This feedback indicates a serious production issue, SLA violation, or service degradation that would breach an enterprise support contract'
          )
    ORDER BY rating ASC
""", conn)
print(f"Enterprise escalations requiring immediate action: {len(escalations_df)}")
escalations_df

## Step 6 — Interpretation & SA Tips

**AI_FILTER vs. SQL WHERE:**
- `WHERE rating < 3` catches low ratings but misses urgent 3-star feedback and passes irrelevant 1-star complaints
- `AI_FILTER` catches the *meaning* of urgency regardless of numeric score

**SA tips:**
- Write conditions as clear, specific statements: *"The text describes X"* works better than vague prompts
- AI_FILTER is boolean — for nuanced routing, chain it with `AI_CLASSIFY`
- Use AI_FILTER in a **Dynamic Table** to maintain a continuously refreshed "action required" queue
- Combine with `STREAMS` to trigger alerts when new urgent feedback arrives